# Spatial Crop Modeler — from your own datacubes

Ask for a yield map in plain language — *"potential yield for beans here in 2020"* — and get one,
from weather and soil cubes **you already have**. A Gemini agent loads the
[`spatial-crop-modeler`](https://github.com/CGIAR-Climate-Data-Hub/skills) skill, reads your cubes
to learn the area and the period, settles the crop, cultivar and planting windows with you, then
this notebook runs DSSAT pixel by pixel and draws the maps.

This is the skill's **`with_cubes`** mode. Nothing is downloaded: no CHIRPS, no SoilGrids, no API
keys beyond Gemini. If you need to build cubes from scratch instead, use the skill's
`full_pipeline` mode outside Colab — the downloads are too slow and too large for a hosted runtime.

---

### How to use this notebook

1. **Runtime ▸ Run all.** Setup takes about three minutes.
2. In section **2**, put your cubes in `/content/cubes` (upload, or mount Drive). If they came from
   an older CDH pipeline, the conversion cell there will fix their format.
3. Edit the request in section **5** and answer the agent's questions. Pressing Enter on an empty
   box accepts what it proposes.
4. The maps appear in section **6**.

### What your cubes must look like

| | Weather cube | Soil cube |
|---|---|---|
| Dimensions | `date`, `y`, `x` | `depth`, `y`, `x` |
| Variables | `precipitation` (mm/day), `tmax`, `tmin` (°C), `srad` (MJ/m²/day) | `wv1500`, `wv0033`, `wv0010`, `bdod`, `soc`, `clay`, `silt`, `cfvo`, `phh2o` in SoilGrids native units |
| Coordinates | `EPSG:4326` degrees | the **same grid** as the weather cube |
| Nodata | `NaN` — never a sentinel like `-9999` | `NaN` |
| Time | real `datetime64`, not `'20200101'` strings | `depth` as `'0-5'`, `'5-15'`, … |

Section **2** checks all of this and tells you what is wrong before any simulation starts. Three
traps worth knowing, because each one fails *quietly* rather than loudly:

> **Coordinates in metres.** `ag-cube-cm` stamps `EPSG:4326` onto whatever coordinates it finds
> without reprojecting, so a cube in SoilGrids' native Goode Homolosine ends up reporting latitude
> 1670931 to DSSAT. Reproject to degrees first.

> **Scaled integers.** The DSSAT writer guesses units — "mean tmax above 100? must be Kelvin,
> subtract 273.15". A cube scaled by 100 sails through that check and produces nonsense. And
> `-9999` is not null, so nodata cells get simulated as real weather.

> **A cube longer than the season you asked for.** The simulation runs *every year present in the
> weather cube*, not just the year you name. A 35-year cube turns one season into 35.


---
## 1 · Setup


In [ ]:
#@title Install ag-cube-cm and the Gemini SDK { display-mode: "form" }
# ag-cube-cm[models] pulls DSSATTools, which ships the DSSAT Fortran binaries.
# aggeodata is NOT needed in with_cubes mode - nothing is downloaded.
%pip install -q "ag-cube-cm[models,viz] @ git+https://github.com/CGIAR-Climate-Data-Hub/ag-cube-cm.git"
%pip install -q -U "google-genai>=2.9.0"
print("Install finished.")

In [ ]:
#@title Environment check { display-mode: "form" }
import importlib.metadata as md_, pathlib


def _version(pkg):
    try:
        return md_.version(pkg)
    except md_.PackageNotFoundError:
        return None


missing = []
for pkg in ("ag-cube-cm", "DSSATTools"):
    v = _version(pkg)
    print(f"{'OK      ' if v else 'MISSING '} {pkg} {v or ''}")
    if not v:
        missing.append(pkg)
if missing:
    raise SystemExit(f"Missing: {', '.join(missing)}. Re-run the install cell; if it still "
                     "fails, use Runtime > Restart session and run all.")

# DSSAT is a Fortran program that fails silently on paths containing spaces.
# Everything here lives under /content, which has none.
CUBE_DIR   = pathlib.Path("/content/cubes")
OUT_DIR    = pathlib.Path("/content/crop_run")
DSSAT_WORK = pathlib.Path("/content/dssat_work")
for d in (CUBE_DIR, OUT_DIR, DSSAT_WORK):
    d.mkdir(parents=True, exist_ok=True)
print(f"\ncubes in   {CUBE_DIR}\noutputs to {OUT_DIR}")

In [ ]:
#@title Connect to the Gemini API { display-mode: "form" }
SECRET_NAME = "GOOGLE_API_KEY" #@param {type:"string"}

import uuid, json
from google import genai
from google.colab import userdata
from IPython.display import display, Markdown
import ipywidgets as widgets

client = genai.Client(api_key=userdata.get(SECRET_NAME))
AGENT  = "antigravity-preview-05-2026"
print("Connected.")

---
## 2 · Your datacubes &nbsp; ✏️ **EDIT ME**

Get the files into `/content/cubes`. Pick whichever route suits you — then run the check cell.


In [ ]:
#@title Upload cubes, or mount Google Drive { display-mode: "form" }
SOURCE = "upload" #@param ["upload", "drive", "already there"]
DRIVE_SUBFOLDER = "cubes" #@param {type:"string"}

import shutil

if SOURCE == "upload":
    from google.colab import files
    print("Select your weather / soil / dem NetCDF files...")
    for name in files.upload():
        shutil.move(name, CUBE_DIR / name)
elif SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    src = pathlib.Path("/content/drive/MyDrive") / DRIVE_SUBFOLDER
    assert src.is_dir(), f"{src} not found - check DRIVE_SUBFOLDER"
    for p in src.glob("*.nc"):
        shutil.copy(p, CUBE_DIR / p.name)

found = sorted(CUBE_DIR.glob("*.nc"))
print(f"\n{len(found)} NetCDF file(s) in {CUBE_DIR}:")
for p in found:
    print(f"  {p.name:42s} {p.stat().st_size/1e6:8.2f} MB")
if not found:
    print("  (none yet - upload, or set SOURCE and re-run)")

### Converting an older cube

Cubes from earlier CDH pipelines are scaled `int32` with `-9999` nodata, in SoilGrids' Goode
Homolosine projection, with a string `date` coordinate. Run this cell to convert them: it masks
nodata to `NaN`, removes the scale factor, reprojects to `EPSG:4326` with **nearest** resampling
(a reprojection, not a regrid — nearest keeps every value exactly as measured), puts the soil cube
on the weather cube's exact grid, and can write a single-season cube so you don't simulate 35
years by accident.

Skip it if your cubes already pass the check below.


In [ ]:
#@title Convert legacy scaled-integer cubes { display-mode: "form" }
RUN_CONVERSION = False #@param {type:"boolean"}
SCALE_FACTOR   = 100 #@param {type:"number"}
NODATA_VALUE   = -9999 #@param {type:"number"}
SEASON_YEAR    = 2020 #@param {type:"number"}
LABEL          = "mycube" #@param {type:"string"}

import numpy as np, pandas as pd, xarray as xr
import rioxarray  # registers the .rio accessor
from rasterio.enums import Resampling

IGH = "+proj=igh +lat_0=0 +lon_0=0 +datum=WGS84 +units=m +no_defs"   # SoilGrids native


def _unscale(ds, scale, nodata, drop=()):
    out = xr.Dataset(coords={k: v for k, v in ds.coords.items() if k != "spatial_ref"})
    for name, da in ds.data_vars.items():
        if name in drop or da.ndim == 0:      # e.g. the 0-d `band` var in DEM files
            continue
        vals = da.values.astype("float32")
        vals[vals == nodata] = np.nan
        out[name] = xr.DataArray(vals / np.float32(scale), dims=da.dims,
                                 coords={d: ds[d] for d in da.dims})
    return out


def _to_wgs84(ds, match=None):
    ds = ds.rio.write_crs(IGH).rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)
    out = (ds.rio.reproject("EPSG:4326", resampling=Resampling.nearest, nodata=np.nan)
           if match is None else
           ds.rio.reproject_match(match, resampling=Resampling.nearest, nodata=np.nan))
    out = out.drop_vars([v for v in ("spatial_ref", "crs") if v in out.variables])
    for name in list(out.data_vars) + list(out.coords):
        for key in ("grid_mapping", "_FillValue"):   # rio.reproject writes both
            out[name].attrs.pop(key, None)
            out[name].encoding.pop(key, None)
    return out


def _save(ds, path):
    for name in list(ds.data_vars) + list(ds.coords):
        for key in ("grid_mapping", "_FillValue"):
            ds[name].attrs.pop(key, None)
            ds[name].encoding.pop(key, None)
    enc = {v: {"zlib": True, "complevel": 4, "dtype": "float32", "_FillValue": np.nan}
           for v in ds.data_vars}
    if path.exists():
        path.unlink()
    ds.to_netcdf(path, encoding=enc)
    print(f"  -> {path.name}  {dict(ds.sizes)}")
    return path


def _find(token):
    hits = sorted(p for p in CUBE_DIR.glob("*.nc")
                  if token in p.name.lower() and LABEL not in p.name)
    return hits[0] if hits else None


if RUN_CONVERSION:
    w_in, s_in, d_in = _find("weather"), _find("soil"), _find("dem")
    assert w_in and s_in, f"need a *weather*.nc and *soil*.nc in {CUBE_DIR}"

    wr = xr.open_dataset(w_in)
    w  = _unscale(wr, SCALE_FACTOR, NODATA_VALUE)
    tdim = next(d for d in ("date", "time") if d in w.dims)
    raw  = wr[tdim].values
    times = (pd.to_datetime(pd.Index(raw), format="%Y%m%d") if raw.dtype.kind in "US"
             else pd.DatetimeIndex(pd.to_datetime(raw)))
    w = w.assign_coords({tdim: (tdim, times.values)})
    if tdim != "date":
        w = w.rename({tdim: "date"})     # `date` is the name both code paths accept
    w = _to_wgs84(w)
    wr.close()
    y0, y1 = int(times.year.min()), int(times.year.max())
    _save(w, CUBE_DIR / f"weather_{LABEL}_{y0}_{y1}.nc")
    if y0 <= SEASON_YEAR <= y1:
        _save(w.sel(date=str(SEASON_YEAR)), CUBE_DIR / f"weather_{LABEL}_{SEASON_YEAR}.nc")

    sr = xr.open_dataset(s_in)
    s  = _to_wgs84(_unscale(sr, SCALE_FACTOR, NODATA_VALUE), match=w)
    if "depth" in s.dims:
        s = s.assign_coords(depth=("depth", [str(v) for v in sr.depth.values]))
    sr.close()
    _save(s, CUBE_DIR / f"soil_{LABEL}.nc")

    if d_in:
        dr = xr.open_dataset(d_in)
        _save(_to_wgs84(_unscale(dr, SCALE_FACTOR, NODATA_VALUE, drop=("band",)), match=w),
              CUBE_DIR / f"dem_{LABEL}.nc")
        dr.close()
    print("\nConversion done - run the check cell next.")
else:
    print("RUN_CONVERSION is off. Tick it if the check below reports format problems.")

---
## 3 · Cube checks and the simulation

`inspect_cubes` runs the format contract, reports the area and period each cube covers, and counts
how many pixels `ag-cube-cm` will actually simulate — using its own mask,
`weather.notnull() & soil.notnull()`, which is why the two cubes must share a grid exactly. It also
estimates the number of DSSAT runs, so a 35-year cube announces itself before it costs you an hour.

`run_sim` writes the `with_cubes` config, runs DSSAT, and applies the skill's Step 7 quality gate —
a clean exit is not a quality signal.


In [ ]:
#@title Inspection, simulation and quality gate { display-mode: "form" }
import time, traceback, urllib.request, urllib.parse
import numpy as np, pandas as pd, xarray as xr
import yaml

MIN_MEAN_HWAM = {"Bean": 150.0, "Maize": 200.0, "Wheat": 200.0}
DSSAT_WEATHER = ("precipitation", "tmax", "tmin", "srad")
DSSAT_SOIL    = ("wv1500", "wv0033", "wv0010", "bdod", "soc", "clay", "silt", "cfvo", "phh2o")

state = {}


def _say(msg):
    try:
        with out:
            print(msg)
    except NameError:
        print(msg)


def _place_name(lat, lon):
    """Reverse-geocode the cube centre so the agent can name the area."""
    try:
        url = "https://nominatim.openstreetmap.org/reverse?" + urllib.parse.urlencode(
            {"lat": lat, "lon": lon, "format": "jsonv2", "zoom": 12, "addressdetails": 1})
        req = urllib.request.Request(url, headers={"User-Agent": "cdh-crop-notebook/1.0"})
        with urllib.request.urlopen(req, timeout=20) as r:
            return json.loads(r.read()).get("display_name")
    except Exception:
        return None


def inspect_cubes(**_):
    """Check every cube in CUBE_DIR against the with_cubes contract."""
    files = sorted(CUBE_DIR.glob("*.nc"))
    if not files:
        return {"state": "no_cubes", "note": f"no .nc files in {CUBE_DIR}"}

    weather, soil, dem, problems = [], [], [], []
    for p in files:
        ds = xr.open_dataset(p)
        dims = set(ds.dims)
        entry = {"file": p.name, "dims": {k: int(v) for k, v in ds.sizes.items()},
                 "variables": list(ds.data_vars)}
        if {"spatial_ref", "crs"} & set(ds.data_vars):
            problems.append(f"{p.name}: spatial_ref/crs is a data variable - drop it, "
                            "ag-cube-cm would use it as the validity mask")
        for v in ds.data_vars:
            if np.issubdtype(ds[v].dtype, np.number) and bool(np.any(ds[v].values == -9999)):
                problems.append(f"{p.name}: {v} still contains -9999; nodata must be NaN")
                break
        if "y" in dims and abs(float(ds.y.min())) > 90:
            problems.append(f"{p.name}: y={float(ds.y.min()):.0f} looks like metres, not "
                            "degrees - reproject to EPSG:4326")
        if "y" in dims and "x" in dims:
            entry["bbox"] = [round(float(ds.x.min()), 4), round(float(ds.y.min()), 4),
                             round(float(ds.x.max()), 4), round(float(ds.y.max()), 4)]

        tdim = next((d for d in ("date", "time") if d in dims), None)
        if tdim:
            if ds[tdim].dtype.kind != "M":
                problems.append(f"{p.name}: {tdim} is {ds[tdim].dtype}, not datetime64")
                years = []
            else:
                years = sorted({int(pd.Timestamp(t).year) for t in ds[tdim].values})
            missing = [v for v in DSSAT_WEATHER if v not in ds.data_vars]
            if missing:
                problems.append(f"{p.name}: weather cube missing {missing}")
            entry.update(kind="weather", time_dim=tdim, years=years,
                         n_days=int(ds.sizes[tdim]))
            if tdim == "time":
                entry["warning"] = ("dim is named 'time'; ag-cube-cm's spatial_cm path "
                                    "hardcodes isel(date=0). Rename to 'date' to be safe.")
            weather.append(entry)
        elif "depth" in dims:
            missing = [v for v in DSSAT_SOIL if v not in ds.data_vars]
            if missing:
                problems.append(f"{p.name}: soil cube missing {missing}")
            entry.update(kind="soil", depths=[str(v) for v in ds.depth.values])
            soil.append(entry)
        else:
            entry.update(kind="dem")
            dem.append(entry)
        ds.close()

    result = {"state": "ok", "weather_cubes": weather, "soil_cubes": soil,
              "dem_cubes": dem, "problems": problems}

    # Grid alignment and the real pixel count, using ag-cube-cm's own mask.
    if weather and soil:
        w = xr.open_dataset(CUBE_DIR / weather[0]["file"])
        s = xr.open_dataset(CUBE_DIR / soil[0]["file"])
        aligned = (np.array_equal(w.y.values, s.y.values)
                   and np.array_equal(w.x.values, s.x.values))
        if not aligned:
            problems.append(
                f"{weather[0]['file']} and {soil[0]['file']} are on different grids. "
                "ag-cube-cm ANDs their validity masks, so this yields zero pixels - "
                "reproject_match the soil cube onto the weather cube.")
        first_w = list(w.data_vars)[0]
        first_s = list(s.data_vars)[0]
        valid = (w[first_w].isel({weather[0]["time_dim"]: 0}).notnull()
                 & s[first_s].isel(depth=0).notnull()) if aligned else None
        n_valid = int(valid.values.sum()) if valid is not None else 0
        result["grids_aligned"] = aligned
        result["valid_pixels"] = n_valid
        result["resolution_deg"] = round(abs(float(w.x[1] - w.x[0])), 6)
        lat = (float(w.y.min()) + float(w.y.max())) / 2
        lon = (float(w.x.min()) + float(w.x.max())) / 2
        result["place"] = _place_name(lat, lon)
        for entry in weather:
            yrs = entry.get("years") or []
            entry["estimated_dssat_runs_per_window"] = len(yrs) * n_valid
        w.close(); s.close()

    result["state"] = "problems" if problems else "ok"
    return result


def run_sim(weather_file, soil_file, crop, cultivar, planting_date,
            dem_file=None, n_planting_windows=4, planting_window_days=14,
            suffix="run", max_pixels=None, ncores=2,
            country="Honduras", country_code="HND", **_):
    """Write the with_cubes config, run DSSAT, then gate the result."""
    try:
        return _run_sim(weather_file, soil_file, crop, cultivar, planting_date, dem_file,
                        n_planting_windows, planting_window_days, suffix, max_pixels,
                        ncores, country, country_code)
    except Exception as exc:
        _say(f"\nSimulation error: {exc}")
        return {"state": "error", "error": f"{type(exc).__name__}: {exc}",
                "traceback": traceback.format_exc()[-1500:]}


def _run_sim(weather_file, soil_file, crop, cultivar, planting_date, dem_file,
             n_planting_windows, planting_window_days, suffix, max_pixels, ncores,
             country, country_code):
    t0 = time.time()
    weather = CUBE_DIR / weather_file
    soil    = CUBE_DIR / soil_file
    for p in (weather, soil):
        if not p.exists():
            return {"state": "missing_file", "note": f"{p} not found"}
    yield_nc = OUT_DIR / f"yield_{suffix}.nc"
    cfg_path = OUT_DIR / f"dssat_{suffix}.yaml"

    cfg = {
        "GENERAL_INFO": {"country": country, "country_code": country_code,
                         "model": "dssat", "working_path": str(DSSAT_WORK),
                         "dssat_path": None, "ncores": int(ncores)},
        # feature=None keeps ag-cube-cm from downloading admin boundaries: the
        # cube already *is* the region of interest.
        "SPATIAL_INFO": {"feature_name": "shapeName", "adm_level": 2, "feature": None,
                         "weather_path": str(weather), "soil_path": str(soil),
                         "output_path": str(yield_nc),
                         "dem_path": str(CUBE_DIR / dem_file) if dem_file else None},
        "CROP": {"name": crop, "cultivar": cultivar},
        "MANAGEMENT": {"planting_date": planting_date,
                       "n_planting_windows": int(n_planting_windows),
                       "planting_window_days": int(planting_window_days)},
    }
    cfg_path.write_text(yaml.dump(cfg, sort_keys=False), encoding="utf-8")
    _say(f"config -> {cfg_path.name}")
    _say(f"DSSAT: {crop} {cultivar}, planting {planting_date}, "
         f"{n_planting_windows} window(s) ...")

    from ag_cube_cm.mcp_server import run_simulation
    res = json.loads(run_simulation(config_path=str(cfg_path), max_pixels=max_pixels))
    if res.get("status") != "ok":
        _say(f"  DSSAT reported: {res.get('message')}")
        return {"state": "simulation_failed", "message": res.get("message"),
                "config": str(cfg_path)}

    out_path = pathlib.Path(res.get("output_path", yield_nc))
    gate = quality_gate(out_path, crop)
    state.update(yield_nc=out_path, weather_nc=weather, soil_nc=soil, crop=crop,
                 cultivar=cultivar, suffix=suffix, planting_date=planting_date,
                 planting_window_days=int(planting_window_days))
    _say(f"\nfinished in {(time.time()-t0)/60:.1f} min - quality gate {gate['verdict']}")
    return {"state": "ok", "simulation": {k: v for k, v in res.items() if k != "message"},
            "quality_gate": gate, "paths": {"yield": str(out_path),
                                            "config": str(cfg_path)}}


def quality_gate(yield_path, crop="Maize", **_):
    """Step 7 of the skill. A clean DSSAT exit is not a quality signal."""
    ds = xr.open_dataset(str(yield_path))
    total     = int(ds.flag.size)
    n_ok      = int((ds.flag.values == 0).sum())
    n_failed  = int((ds.flag.values == 1).sum())
    n_no_data = int((ds.flag.values == 2).sum())
    mean_hwam = float(np.nanmean(ds.HWAM.where(ds.flag == 0).values)) if n_ok else 0.0
    ds.close()

    ok_frac   = n_ok / total if total else 0.0
    fail_frac = n_failed / max(n_ok + n_failed, 1)
    floor     = MIN_MEAN_HWAM.get(crop, 200.0)
    checks = {"ok_fraction":   (round(ok_frac, 3),   ok_frac   >= 0.20),
              "fail_fraction": (round(fail_frac, 3), fail_frac <= 0.50),
              "mean_hwam":     (round(mean_hwam, 1), mean_hwam >= floor)}
    passed = all(p for _, p in checks.values())
    return {"verdict": "PASSED" if passed else "FAILED",
            "pixels": {"ok": n_ok, "failed": n_failed, "no_data": n_no_data,
                       "total": total},
            "mean_hwam_kg_ha": round(mean_hwam, 1), "hwam_floor_used": floor,
            "checks": checks,
            "diagnosis": None if passed else (
                "Probable causes, ordered: (1) wrong planting season for this region; "
                "(2) cultivar incompatible with the climate; (3) soil and weather grids "
                "misaligned so few pixels carry both; (4) the cube covers water or bare "
                "rock.")}


print("Inspection and simulation loaded.")

In [ ]:
#@title Plotting - multitemporal yield and climate panels { display-mode: "form" }
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 10, "font.size": 9})


def _panels(n):
    cols = min(n, 4)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(3.4 * cols, 3.1 * rows), squeeze=False)
    return fig, [a for row in axes for a in row]


def plot_yield_maps():
    """One map per planting window per year, plus the mean across all of them."""
    ds   = xr.open_dataset(str(state["yield_nc"]))
    hwam = ds["HWAM"].where(ds["flag"] == 0)          # flag 0 = DSSAT produced output
    pw   = [int(v) for v in np.atleast_1d(hwam["planting_window"].values)]
    yrs  = [int(v) for v in np.atleast_1d(hwam["year"].values)]

    frames, stats = [], []
    for y in yrs:
        for w in pw:
            sl  = hwam.sel(year=y, planting_window=w)
            day = (np.datetime64(state["planting_date"])
                   + np.timedelta64(w * state["planting_window_days"], "D"))
            vals = sl.values[~np.isnan(sl.values)]
            frames.append((f"{state['crop']} {y}\nplanted {str(day)[:10]}", sl))
            stats.append({"year": y, "planting_window": w, "planting_date": str(day)[:10],
                          "pixels_with_yield": int(vals.size),
                          "mean_kg_ha": round(float(vals.mean()), 1) if vals.size else None,
                          "min_kg_ha":  round(float(vals.min()), 1)  if vals.size else None,
                          "max_kg_ha":  round(float(vals.max()), 1)  if vals.size else None})

    allv = hwam.values[~np.isnan(hwam.values)]
    vmax = float(np.nanpercentile(allv, 98)) if allv.size else 1.0

    fig, axes = _panels(len(frames) + 1)
    for ax, (title, sl) in zip(axes, frames):
        sl.plot(ax=ax, cmap="YlGn", vmin=0, vmax=vmax, add_colorbar=False)
        ax.set_title(title); ax.set_xlabel(""); ax.set_ylabel("")
    mean = hwam.mean(dim=["planting_window", "year"])
    im   = mean.plot(ax=axes[len(frames)], cmap="YlGn", vmin=0, vmax=vmax,
                     add_colorbar=False)
    axes[len(frames)].set_title("mean of all windows")
    axes[len(frames)].set_xlabel(""); axes[len(frames)].set_ylabel("")
    for ax in axes[len(frames) + 1:]:
        ax.axis("off")
    fig.colorbar(im, ax=axes, shrink=0.7, label="HWAM (kg/ha)")
    fig.suptitle(f"{state['crop']} {state['cultivar']} yield - {state['suffix']}", y=1.02)
    png = OUT_DIR / f"yield_maps_{state['suffix']}.png"
    fig.savefig(png, bbox_inches="tight"); plt.show()
    ds.close()
    return {"panels": stats, "png": str(png), "colour_scale_max_kg_ha": round(vmax, 1)}


def plot_climate_maps():
    """Monthly rainfall total and mean daily max temperature from the weather cube."""
    ds = xr.open_dataset(str(state["weather_nc"]))
    tdim = next((d for d in ("date", "time") if d in ds.dims), "date")
    summaries = {}
    for var, how, cmap, label in [("precipitation", "sum", "Blues", "monthly rainfall (mm)"),
                                  ("tmax", "mean", "YlOrRd", "mean daily Tmax (degC)")]:
        if var not in ds.data_vars:
            continue
        monthly = getattr(ds[var].resample({tdim: "1MS"}), how)()
        n = int(monthly.sizes[tdim])
        fig, axes = _panels(n)
        vmin = float(np.nanpercentile(monthly.values, 2))
        vmax = float(np.nanpercentile(monthly.values, 98))
        im = None
        for ax, t in zip(axes, monthly[tdim].values):
            im = monthly.sel({tdim: t}).plot(ax=ax, cmap=cmap, vmin=vmin, vmax=vmax,
                                             add_colorbar=False)
            ax.set_title(str(t)[:7]); ax.set_xlabel(""); ax.set_ylabel("")
        for ax in axes[n:]:
            ax.axis("off")
        if im is not None:
            fig.colorbar(im, ax=axes, shrink=0.7, label=label)
        fig.suptitle(f"{label} - {state['suffix']}", y=1.02)
        png = OUT_DIR / f"climate_{var}_{state['suffix']}.png"
        fig.savefig(png, bbox_inches="tight"); plt.show()
        summaries[var] = {
            "by_month": {str(t)[:7]: round(float(np.nanmean(monthly.sel({tdim: t}).values)), 1)
                         for t in monthly[tdim].values},
            "png": str(png)}
    ds.close()
    return summaries


def plot_maps(kind="both", **_):
    if not state.get("yield_nc"):
        return {"state": "nothing_to_plot", "note": "no simulation result yet"}
    result = {"state": "ok"}
    if kind in ("both", "climate"):
        result["climate"] = plot_climate_maps()
    if kind in ("both", "yield"):
        result["yield"] = plot_yield_maps()
    return result


print("Plotting loaded.")

In [ ]:
#@title Tool declarations { display-mode: "form" }
ask_user_tool = {
    "type": "function",
    "name": "ask_user",
    "description": (
        "Ask the user for a BATCH of parameters in a single call and receive every answer "
        "together. Put EVERY field you still need into `questions` - the client presents them "
        "one at a time and returns all the answers in one response, so a batch of 10 costs "
        "exactly what a batch of 1 costs. Calling this with one question and waiting burns the "
        "user's quota and is the most common way this workflow fails."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "questions": {
                "type": "array",
                "description": "Every parameter you need right now. Longer is better.",
                "items": {
                    "type": "object",
                    "properties": {
                        "field":      {"type": "string", "description": "Parameter, e.g. 'planting_date'."},
                        "question":   {"type": "string", "description": "The question, in plain language."},
                        "suggestion": {"type": "string", "description": "Draft value the user accepts with Enter."},
                    },
                    "required": ["field", "question"],
                },
            },
        },
        "required": ["questions"],
    },
}

inspect_cubes_tool = {
    "type": "function",
    "name": "inspect_cubes",
    "description": (
        "Read every datacube the user provided and report what they contain: dimensions, "
        "variables, bounding box, the reverse-geocoded place name, the years each weather cube "
        "covers, the soil depths, whether the grids are aligned, how many pixels will actually "
        "simulate, and any format violations. Runs in the notebook - your sandbox cannot see "
        "these files. Call this FIRST, before asking the user anything: the cubes already answer "
        "where and when, so those are not questions. If `problems` is non-empty, report them and "
        "stop - point the user at the conversion cell in section 2 rather than simulating."
    ),
    "parameters": {"type": "object", "properties": {}, "required": []},
}

run_sim_tool = {
    "type": "function",
    "name": "run_sim",
    "description": (
        "Run DSSAT over every valid pixel of the chosen cubes and apply the skill's quality "
        "gate. Runs in the notebook; your sandbox has no DSSAT binary. Call it ONCE, after the "
        "user confirms the plan. Pick `weather_file` deliberately: the simulation runs every "
        "year present in that cube, so choose a single-season cube unless the user wants a "
        "multi-year run. Returns the pixel summary and the gate verdict; if the gate FAILED, "
        "report the diagnosis and stop instead of plotting."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "weather_file": {"type": "string", "description": "Weather cube filename from inspect_cubes."},
            "soil_file":    {"type": "string", "description": "Soil cube filename from inspect_cubes."},
            "dem_file":     {"type": "string", "description": "Optional DEM filename; improves the .WTH elevation."},
            "crop":         {"type": "string", "description": "Bean, Maize or Wheat."},
            "cultivar":     {"type": "string", "description": "DSSAT cultivar code, e.g. IB0001 for Bean."},
            "planting_date": {"type": "string", "description": "YYYY-MM-DD, first planting window."},
            "n_planting_windows":   {"type": "integer", "description": "Windows to simulate (4 gives four maps)."},
            "planting_window_days": {"type": "integer", "description": "Days between windows (default 14)."},
            "suffix":       {"type": "string", "description": "Short label, no spaces, e.g. 'beans_yoro_2020'."},
            "max_pixels":   {"type": "integer", "description": "Cap for a quick test; omit to run every pixel."},
            "ncores":       {"type": "integer", "description": "Workers (default 2)."},
            "country":      {"type": "string", "description": "Country name, e.g. Honduras."},
            "country_code": {"type": "string", "description": "ISO3, e.g. HND."},
        },
        "required": ["weather_file", "soil_file", "crop", "cultivar", "planting_date", "suffix"],
    },
}

plot_maps_tool = {
    "type": "function",
    "name": "plot_maps",
    "description": (
        "Draw the multitemporal maps in the notebook after the quality gate passes. "
        "kind='yield' gives one panel per planting window (and per year, when several were "
        "simulated); kind='climate' gives monthly rainfall and max-temperature panels from the "
        "weather cube; kind='both' draws both. Call once, at the end. Returns the numbers behind "
        "each panel so you can describe them - you cannot see the images."
    ),
    "parameters": {
        "type": "object",
        "properties": {"kind": {"type": "string", "description": "'yield', 'climate' or 'both'."}},
        "required": [],
    },
}

TOOLS = [ask_user_tool, inspect_cubes_tool, run_sim_tool, plot_maps_tool]
print("Tools declared:", [t["name"] for t in TOOLS])

In [ ]:
#@title Chat plumbing - the question box and the turn loop { display-mode: "form" }
import re

MAX_RETRIES = 4

out  = widgets.Output()
box  = widgets.Text(layout=widgets.Layout(width="80%"))
send = widgets.Button(description="Send", button_style="primary")
ui   = widgets.HBox([box, send])

state.update({"result": None, "env": None, "queue": [], "answers": [],
              "subq": None, "collected": {}, "batch": 0, "turns": 0})

LOCAL = {"inspect_cubes": inspect_cubes, "run_sim": run_sim, "plot_maps": plot_maps}


def api_call(**kwargs):
    """interactions.create with backoff, so a free-tier 429 pauses instead of crashing."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            res = client.interactions.create(**kwargs)
            state["turns"] += 1
            return res
        except Exception as exc:
            text = str(exc)
            if "429" not in text and "too_many_requests" not in text.lower():
                raise
            found = re.search(r"retry in ([\d.]+)s", text)
            wait  = float(found.group(1)) + 2 if found else 20 * attempt
            with out:
                print(f"rate limited - waiting {wait:.0f}s "
                      f"(attempt {attempt}/{MAX_RETRIES}; the notebook will resume on its own)")
            time.sleep(wait)
    raise RuntimeError("Still rate limited after several retries - wait a minute and re-run, "
                       "or use a paid key.")


def show(obj, prefix="**Agent**\n\n"):
    text = obj if isinstance(obj, str) else getattr(obj, "output_text", None)
    if text:
        display(Markdown(prefix + text))


def pending_calls(res):
    done = {s.call_id for s in res.steps if s.type == "function_result"}
    return [s for s in res.steps if s.type == "function_call" and s.id not in done]


def call_args(call):
    return call.arguments if isinstance(call.arguments, dict) else json.loads(call.arguments)


def reply(call, payload):
    state["answers"].append({"type": "function_result", "name": call.name,
                             "call_id": call.id, "result": payload})


def end(msg):
    box.disabled = send.disabled = True
    with out:
        display(Markdown(f"### {msg}"))


def run_local(call):
    """Execute a non-interactive tool call here in the notebook."""
    fn = LOCAL.get(call.name)
    if fn is None:
        return {"error": f"no local handler for {call.name}"}
    with out:
        display(Markdown(f"*running `{call.name}`...*"))
    res = fn(**call_args(call))
    with out:
        if call.name == "inspect_cubes" and res.get("state") in ("ok", "problems"):
            lines = []
            for c in res.get("weather_cubes", []):
                yrs = c.get("years") or []
                span = f"{yrs[0]}-{yrs[-1]}" if yrs else "?"
                lines.append(f"- weather `{c['file']}` {c['n_days']} days, {span}, "
                             f"{len(c['variables'])} vars")
            for c in res.get("soil_cubes", []):
                lines.append(f"- soil `{c['file']}` depths {c['depths']}")
            for c in res.get("dem_cubes", []):
                lines.append(f"- dem `{c['file']}`")
            if res.get("place"):
                lines.append(f"- covers **{res['place']}**")
            if res.get("valid_pixels") is not None:
                lines.append(f"- **{res['valid_pixels']} simulatable pixels** at "
                             f"{res.get('resolution_deg')} deg")
            for p in res.get("problems", []):
                lines.append(f"- FORMAT PROBLEM: {p}")
            display(Markdown("\n".join(lines)))
        elif call.name == "run_sim" and res.get("quality_gate"):
            g = res["quality_gate"]
            display(Markdown(
                f"**Quality gate {g['verdict']}** - pixels ok {g['pixels']['ok']}/"
                f"{g['pixels']['total']}, failed {g['pixels']['failed']}, "
                f"no_data {g['pixels']['no_data']}, mean HWAM {g['mean_hwam_kg_ha']} kg/ha"))
    return res


def questions_of(call):
    args  = call_args(call)
    items = args.get("questions")
    if isinstance(items, list) and items:
        return [q for q in items if isinstance(q, dict) and q.get("question")]
    if args.get("question"):
        return [{"field": args.get("field", "value"), "question": args["question"],
                 "suggestion": args.get("suggestion")}]
    return []


def pose(q):
    done    = state["batch"] - len(state["subq"]) + 1
    counter = f"`{done}/{state['batch']}` " if state["batch"] > 1 else ""
    text    = f"{counter}**{q.get('field', 'question')}** - {q['question']}"
    if q.get("suggestion"):
        text += (f"\n\n> *Proposed:* {q['suggestion']}\n>\n"
                 "> *Press Enter on an empty box to accept, or type a replacement.*")
    with out:
        display(Markdown(text))
    box.placeholder = "Enter to accept" if q.get("suggestion") else "Your answer..."
    box.disabled = send.disabled = False
    box.value = ""


def advance_turn():
    box.disabled = send.disabled = True
    with out:
        print(f"thinking... (turn {state['turns'] + 1}, sending {len(state['answers'])} result(s))")
    state["result"] = api_call(agent=crop_agent.id,
                               previous_interaction_id=state["result"].id,
                               environment=state["env"], input=state["answers"], tools=TOOLS)
    state["answers"] = []
    with out:
        show(state["result"])
    ask_next()


def ask_next():
    """Drive the queue: run local tools silently, pause only for questions."""
    while True:
        while state["queue"] and state["queue"][0].name != "ask_user":
            call = state["queue"].pop(0)
            reply(call, run_local(call))

        if state["queue"]:
            call = state["queue"][0]
            if state["subq"] is None:
                state["subq"]      = questions_of(call)
                state["collected"] = {}
                state["batch"]     = len(state["subq"])
                if state["batch"] > 1:
                    with out:
                        display(Markdown(f"*{state['batch']} questions in this batch - answering "
                                         "all of them costs a single API turn.*"))
            if state["subq"]:
                return pose(state["subq"][0])
            state["queue"].pop(0)
            reply(call, {"answers": state["collected"]})
            state["subq"] = None
            continue

        if state["answers"]:
            return advance_turn()

        res = state["result"]
        if res.status != "requires_action":
            return end(f"Finished in {state['turns']} API turns.")
        calls = pending_calls(res)
        if not calls:
            return end(f"Finished in {state['turns']} API turns - nothing left to answer.")
        state["queue"] = calls


def submit(_):
    if not state["subq"] or box.disabled:
        return
    msg, box.value = box.value.strip(), ""
    q = state["subq"].pop(0)
    answer = msg or q.get("suggestion") or ""
    with out:
        display(Markdown(f"**You** - {answer}"))
    state["collected"][q.get("field", f"q{len(state['collected'])}")] = answer
    ask_next()


send.on_click(submit)
box.on_submit(submit)   # Enter key (deprecated in ipywidgets 8 - harmless warning)


def start(first_input):
    out.clear_output()
    state.update(result=None, env=None, queue=[], answers=[], subq=None,
                 collected={}, batch=0, turns=0)
    state["result"] = api_call(agent=crop_agent.id, input=first_input,
                               environment="remote", tools=TOOLS)
    state["env"] = state["result"].environment_id
    display(out, ui)
    with out:
        show(state["result"])
    ask_next()


print("Helpers loaded.")

---
## 4 · Agent instructions &nbsp; ✏️ **EDIT ME**

The agent's standing brief, with the full `SKILL.md` appended so it never spends a turn reading a
file. Widen the *PROPOSE* list to be asked fewer questions.


In [ ]:
#@title Build the standing brief { display-mode: "form" }
# Fetch the skill text straight from GitHub - no npx, no sandbox mount needed.
import urllib.request
_SKILL_URL = ("https://raw.githubusercontent.com/CGIAR-Climate-Data-Hub/skills/main/"
              ".agents/skills/spatial-crop-modeler/SKILL.md")
with urllib.request.urlopen(urllib.request.Request(
        _SKILL_URL, headers={"User-Agent": "cdh-crop-notebook/1.0"}), timeout=30) as _r:
    SKILL_TEXT = _r.read().decode("utf-8")

BRIEF = """
You are a spatial crop modelling agent for the CGIAR Climate Data Hub. You run DSSAT over a
region the user already has datacubes for, and explain the result. The full spatial-crop-modeler
skill is appended below. Follow its agronomy and its quality gate, but note that this notebook
runs the skill's `with_cubes` mode only, with four differences.

1. NOTHING IS DOWNLOADED, AND THE NOTEBOOK RUNS THE MODEL. Your sandbox has no data, no internet
   and no DSSAT binary. Ignore the skill's full_pipeline mode, its aggeodata steps and its soil
   cube building entirely - the user already has the cubes. `run_sim` writes the config and runs
   DSSAT; `inspect_cubes` reads the cubes. Never write configs or shell commands yourself, and
   never ask the user to run anything.

2. THE CUBES ANSWER 'WHERE' AND 'WHEN'. Call `inspect_cubes` FIRST, before any question. It
   returns the bounding box, the place name, the years covered, the soil depths and the number of
   simulatable pixels. So never ask the user for a bounding box, a country or a date range, and
   never resolve a place name yourself - if what the user said disagrees with what the cubes
   cover, say so plainly and use the cubes, because they are the data that exists. If
   `inspect_cubes` reports `problems`, list them, point the user at the conversion cell in
   section 2 of the notebook, and stop. Do not simulate a cube that failed its checks.

3. PICK THE WEATHER CUBE DELIBERATELY. The simulation runs every year present in the cube you
   choose, not just the year the user named. When several weather cubes are offered, choose the
   one whose years match the request and say why; when only a long cube exists and the user wants
   one season, warn them what it will cost (the estimated run count is in the inspect_cubes
   result) and let them decide.

4. YOU HAVE NO CHAT CHANNEL. The only way to ask anything is `ask_user`, which takes a LIST of
   questions. A question written as message text will never be answered, so never end a message
   with one. Batch everything: the user pays per turn and every turn resends this conversation.
   Your first ask_user call should carry every parameter you still need. Aim to finish in under
   six turns.

PARAMETERS

  Propose, then confirm: cultivar, planting_date, the number of planting windows, suffix (a short
  slug like beans_yoro_2020), country and country_code (infer them from the place name
  inspect_cubes returned).

  Never guess: the planting date for a region or cropping season you are not confident about.
  Ask, with your best proposal as the `suggestion` so a single Enter accepts it.

  Cultivars: Bean -> IB0001 (CRGRO048). Maize -> IB1072 tropical, PC0002 temperate.
  Wheat -> IB1015 winter, IB0001 spring. Wheat uses WHAPS048; WHCER codes such as IB1487 fail
  with rc=99.

  Planting dates in Central America: beans have two seasons - primera, sown May/June, and
  postrera, sown September. Ask which one the user means rather than assuming; it changes the
  result completely.

  MULTITEMPORAL MEANS PLANTING WINDOWS. One year and one window produce a single map. To show
  yield over time within a season use n_planting_windows: 4 with planting_window_days: 14 - four
  staggered sowings, four maps. State which you are doing.

AFTER THE RUN

  Report the pixel summary verbatim: ok / failed / no_data and mean HWAM. If the quality gate
  FAILED, stop. Show the diagnosis, say which cause you think applies and why, and ask the user
  how to proceed - do not plot, and do not describe a failed run as a result. Only when the gate
  PASSES, call `plot_maps` once. You cannot see the images, so describe the numbers it returns:
  which planting window yielded most, the spread across the area, and where the low pixels are.
  Close with the honest caveats - potential yield with no fertiliser schedule, one cultivar,
  rainfed, and whatever the cubes' own resolution and period imply.

--- FULL SKILL BELOW ---
"""

SYSTEM_INSTRUCTION = BRIEF + "\n" + SKILL_TEXT
print(f"System instruction: {len(SYSTEM_INSTRUCTION):,} chars "
      f"({len(BRIEF):,} brief + {len(SKILL_TEXT):,} skill)")

---
## 5 · Create the agent and make your request &nbsp; ✏️ **EDIT ME**

Re-run the agent cell after editing the brief — it is baked in at creation time.

Then say what you want. You do not need to describe the area: the cubes do that. A crop, a season
and what you want to see is enough.

| You want | Write something like |
|---|---|
| One season, several sowings | `potential yield for beans in 2020, show me how it changes by planting date` |
| A specific cultivar | `maize IB1072 planted 15 May 2020` |
| A quick test first | `bean yield for 2020, but only run 20 pixels so I can check it works` |


In [ ]:
#@title Create the agent { display-mode: "form" }
crop_agent = client.agents.create(
    id=f"spatial-crop-modeler-{uuid.uuid4().hex[:8]}",
    base_agent=AGENT,
    system_instruction=SYSTEM_INSTRUCTION,
    base_environment={"type": "remote"},
)
print(f"Agent ready: {crop_agent.id}")

In [ ]:
PROMPT = "I would like to see the potential yield for beans in San Isidro, Yoro, Honduras for 2020" #@param {type:"string"}

start(PROMPT)

---
## 6 · The maps again

The agent draws the panels during the conversation above. Run this to redraw or export them
without spending another API turn.


In [ ]:
#@title Redraw and list the outputs { display-mode: "form" }
if state.get("yield_nc"):
    summary = plot_maps("both")
    for panel in summary.get("yield", {}).get("panels", []):
        print(f"  {panel['planting_date']}  mean {panel['mean_kg_ha']} kg/ha "
              f"(range {panel['min_kg_ha']}-{panel['max_kg_ha']}, "
              f"{panel['pixels_with_yield']} pixels)")
    print("\nFiles:")
    for key in ("weather_nc", "soil_nc", "yield_nc"):
        print(f"  {key:11s} {state[key]}")
else:
    print("No simulation in this session yet - run the cells above first.")

---
## What next?

- **This is potential yield.** No fertiliser schedule, one cultivar, rainfed, and DSSAT's own
  management assumptions. Treat the numbers as an upper envelope shaped by weather and soil, not a
  forecast of what a farm will harvest.
- **Check the failed pixels before trusting the mean.** A cluster of `flag=2` on one edge usually
  means missing soil or weather, not bad agronomy. The quality gate catches catastrophic cases
  only.
- **Soil depth matters.** A cube that stops at 60 cm gives DSSAT a shallower profile to draw water
  from than one reaching 100 cm, and rainfed yields are sensitive to exactly that.
- **Document the run** with the [`cdh-metadata` notebook](https://colab.research.google.com/github/CGIAR-Climate-Data-Hub/skills/blob/main/notebooks/CDH_metadata_example.ipynb)
  — it turns `yield_<suffix>.nc` into a CDH catalogue record.
- **Read the skill**: [spatial-crop-modeler](https://github.com/CGIAR-Climate-Data-Hub/skills/blob/main/.agents/skills/spatial-crop-modeler/SKILL.md)
  · [troubleshooting](https://github.com/CGIAR-Climate-Data-Hub/skills/blob/main/.agents/skills/spatial-crop-modeler/references/troubleshooting.md)
  · [the converter used in section 2](https://github.com/CGIAR-Climate-Data-Hub/skills/blob/main/scripts/convert_legacy_cubes.py)
  · [ag-cube-cm](https://github.com/CGIAR-Climate-Data-Hub/ag-cube-cm)
- **More skills**: [CGIAR-Climate-Data-Hub/skills](https://github.com/CGIAR-Climate-Data-Hub/skills)
